# Mario Kart Tracker v2: Data Scraping
## Vehicles

The aim of this notebook is to obtain the vehicles for the games and prepare an appropriate CSV file that can then be uploaded to the database.

We will source the vehicles from the Mario Wiki pages. 

Because MK 8 Deluxe has the options for choosing wheels and gliders, we will include those here too (as separate files)

Required libraries:
* `Pandas`
* `BeautifulSoup`
* `Selenium`

In [1]:
# Import required libraries
import pandas as pd
from bs4 import BeautifulSoup, NavigableString, Tag
from selenium import webdriver
from selenium.webdriver.common.by import By
from pathlib import Path

In [2]:
# Set the working directory
wd = Path().cwd()
wd = wd.parent

### Mario Kart 8 Deluxe

Let's start with MK 8. 

We basically need everything on [this page](https://mariokart.fandom.com/wiki/Mario_Kart_8_Deluxe#Vehicle_Parts) in the *Vehicle Parts* section.

For the most part, these are links, though some have an alternative name - this appears to be the name used in Australia (and Europe), so this is what we want for our purposes.


In [3]:
url = "https://mariokart.fandom.com/wiki/Mario_Kart_8_Deluxe"

driver = webdriver.Firefox()
driver.get(url)

mk8_html = driver.page_source

mk8_soup = BeautifulSoup(mk8_html)

driver.quit()

In [4]:
import re

In [5]:
elements = mk8_soup.find_all("h2")
for elem in elements:
    if re.match(r"Vehicle Parts", elem.text):
        section_heading = elem
        break

next_elements = elem.find_all_next(re.compile("h2|h3|table"))
section = None
body_tables = []
glider_tables = []
wheel_tables = []

for elem in next_elements:
    if elem.name == "h3":
        if "Bodies" in elem.text:
            section = "Bodies"
        elif "Wheels" in elem.text:
            section = "Wheels"
        elif "Gliders" in elem.text:
            section = "Gliders"
    elif elem.name == "table":
        if section == "Bodies":
            body_tables.append(elem)
        elif section == "Wheels":
            wheel_tables.append(elem)
        elif section == "Gliders":
            glider_tables.append(elem)
    else:
        break

body_tables

[<table style="text-align:center; margin:auto; font-weight:bold; width:100%">
 <caption>
 </caption>
 <tbody><tr>
 <td width="17%"><span typeof="mw:File"><a href="/wiki/Standard_Kart" title="Standard Kart"><img alt="StandardKartBodyMK8" class="mw-file-element lazyload" data-image-key="StandardKartBodyMK8.png" data-image-name="StandardKartBodyMK8.png" data-relevant="0" data-src="https://static.wikia.nocookie.net/mariokart/images/0/05/StandardKartBodyMK8.png/revision/latest/scale-to-width-down/100?cb=20140715154926" decoding="async" height="64" loading="lazy" src="data:image/gif;base64,R0lGODlhAQABAIABAAAAAP///yH5BAEAAAEALAAAAAABAAEAQAICTAEAOw%3D%3D" width="100"/></a></span>
 </td>
 <td width="17%"><span typeof="mw:File"><a href="/wiki/Pipe_Frame" title="Pipe Frame"><img alt="PipeFrameBodyMK8" class="mw-file-element lazyload" data-image-key="PipeFrameBodyMK8.png" data-image-name="PipeFrameBodyMK8.png" data-relevant="0" data-src="https://static.wikia.nocookie.net/mariokart/images/d/d1/Pip

In [6]:
# For each row of each table, we want to check whether it contains images. We don't want the image rows
# These are all contained in <span> tags, so we can look at the first child of each row and then exclude the row if it is a span

rows = []

for table in body_tables:
    data = table.find("tbody")
    for row in data.find_all("tr"):
        if row:
            child = row.find("td")
            grandchild = child.find()
            if grandchild:
                if grandchild.name != "span" and grandchild.name != "div":
                    rows.append(row)
            else:
                rows.append(row)

print(rows)

[<tr>
<td><a href="/wiki/Standard_Kart" title="Standard Kart">Standard Kart</a>*/**
</td>
<td><a href="/wiki/Pipe_Frame" title="Pipe Frame">Pipe Frame</a>*
</td>
<td><a href="/wiki/Mach_8" title="Mach 8">Mach 8</a>
</td>
<td><a href="/wiki/Steel_Driver" title="Steel Driver">Steel Driver</a>**
</td>
<td><a href="/wiki/Cat_Cruiser" title="Cat Cruiser">Cat Cruiser</a>
</td>
<td><a href="/wiki/Circuit_Special" title="Circuit Special">Circuit Special</a>*/**
</td></tr>, <tr>
<td height="25px">
</td></tr>, <tr>
<td><a href="/wiki/Tri-Speeder" title="Tri-Speeder">Tri-Speeder</a>
</td>
<td><a href="/wiki/Badwagon" title="Badwagon">Badwagon</a>
</td>
<td>
<p><a href="/wiki/Prancer" title="Prancer">Prancer</a>
</p>
</td>
<td><a href="/wiki/Biddybuggy" title="Biddybuggy">Biddybuggy</a>*
</td>
<td><a href="/wiki/Landship" title="Landship">Landship</a>*
</td>
<td><a href="/wiki/Sneeker" title="Sneeker">Sneeker</a>*/**
</td></tr>, <tr>
<td>
</td>
<td>
</td>
<td>
</td>
<td><small><i>Buggybud</i></sma

In [7]:
import numpy as np

In [8]:
row = rows[3]

for child in row.children:
    if isinstance(child, NavigableString):
        continue
    elif isinstance(child, Tag):
        small = child.select('small')
        print(small)
    

[]
[]
[]
[<small><i>Buggybud</i></small>]
[]
[<small><i>Bounder</i></small>]


In [9]:
rows_simplified = []

for row in rows:
    row_simplified = []
    for child in row.children:
        if isinstance(child, NavigableString):
            continue
        
        # If there is an alternative name, it will be in small italics
        # Check for this
        small = child.select('small')
        if small:
            value = child.text.rstrip("\n*/ ").lstrip("\n ") + " [AUS]"
        else:
            value = child.text.rstrip("\n*/ ").lstrip("\n ")

        if value in {"", "New"}:
            row_simplified.append(np.nan)
        else:
            row_simplified.append(value)
    
    rows_simplified.append(row_simplified)

body_df = pd.DataFrame(rows_simplified)

body_df = body_df.dropna(axis=1, how="all")
body_df = body_df.dropna(axis=0, how="all")

body_df = body_df.reset_index(drop=True)

body_df

,0,1,2,3,4,5
0,Standard Kart,Pipe Frame,Mach 8,Steel Driver,Cat Cruiser,Circuit Special
1,Tri-Speeder,Badwagon,Prancer,Biddybuggy,Landship,Sneeker
2,NaN,NaN,NaN,Buggybud [AUS],NaN,Bounder [AUS]
3,Sports Coupe,Gold Standard,GLA,W 25 Silver Arrow,300 SL Roadster,Blue Falcon
4,NaN,Gold Kart [AUS],NaN,NaN,NaN,NaN
5,Tanooki Kart,B-Dasher,Streetle,P-Wing,Koopa Clown,NaN
6,Tanooki Buggy [AUS],NaN,NaN,NaN,NaN,NaN
7,Standard Bike,Comet,Sport Bike,The Duke,Flame Rider,Varmint
8,Mr. Scooty,Jet Bike,Yoshi Bike,Master Cycle,Master Cycle Zero,City Tripper
9,Mr Scooty [AUS],NaN,NaN,NaN,NaN,NaN


In [10]:
import math

In [27]:
wrangled_df = body_df.copy()

for row in range(0, len(body_df) - 1):
    next_row_contains_aus_names = False
    for col in wrangled_df.columns:
        value = wrangled_df.loc[row, col]
        next_value = wrangled_df.loc[row + 1, col]

        if isinstance(value, str) and isinstance(next_value, str):
            if "[AUS]" in next_value:
                next_row_contains_aus_names = True
                wrangled_df.loc[row, col] = next_value.replace("[AUS]", "")

    # Fill the first row with "False", since it won't have Aus. names
    if row == 0:
        wrangled_df.loc[row, "Contains Aus. Names"] = False

    wrangled_df.loc[row + 1, "Contains Aus. Names"] = next_row_contains_aus_names

wrangled_df = wrangled_df[wrangled_df["Contains Aus. Names"] == False]
wrangled_df = wrangled_df.drop(columns=["Contains Aus. Names"])

body_values = [list(row) for row in list(wrangled_df.values)]
body_values = list(set([val for row in body_values for val in row]))

body_final_df = pd.DataFrame(body_values)
body_final_df = body_final_df[body_final_df[0].notna()]
body_final_df


,0
0,Teddy Buggy
1,Tri-Speeder
3,Bounder
4,Varmint
5,Sport Bike
6,Mr Scooty
7,Master Cycle
8,Comet
9,Buggybud
10,Pipe Frame


### Wheels and Gliders

Now that we have the bodies, let's sort out the wheels and gliders.

We can largely do the same as we did for the bodies, as the sections are structured quite similarly.

In [29]:
rows = []
rows_simplified = []

for table in glider_tables:
    data = table.find("tbody")
    for row in data.find_all("tr"):
        if row:
            child = row.find("td")
            grandchild = child.find()
            if grandchild:
                if grandchild.name != "span" and grandchild.name != "div":
                    rows.append(row)
            else:
                rows.append(row)

for row in rows:
    row_simplified = []
    for child in row.children:
        if isinstance(child, NavigableString):
            continue
        
        # If there is an alternative name, it will be in small italics
        # Check for this
        small = child.select('small')
        if small:
            value = child.text.rstrip("\n*/ ").lstrip("\n ") + " [AUS]"
        else:
            value = child.text.rstrip("\n*/ ").lstrip("\n ")

        if value in {"", "New"}:
            row_simplified.append(np.nan)
        else:
            row_simplified.append(value)
    
    rows_simplified.append(row_simplified)

glider_df = pd.DataFrame(rows_simplified)

glider_df = glider_df.dropna(axis=1, how="all")
glider_df = glider_df.dropna(axis=0, how="all")

glider_df = glider_df.reset_index(drop=True)

wrangled_df = glider_df.copy()

for row in range(0, len(glider_df) - 1):
    next_row_contains_aus_names = False
    for col in wrangled_df.columns:
        value = wrangled_df.loc[row, col]
        next_value = wrangled_df.loc[row + 1, col]

        if isinstance(value, str) and isinstance(next_value, str):
            if "[AUS]" in next_value:
                next_row_contains_aus_names = True
                wrangled_df.loc[row, col] = next_value.replace("[AUS]", "")

    # Fill the first row with "False", since it won't have Aus. names
    if row == 0:
        wrangled_df.loc[row, "Contains Aus. Names"] = False

    wrangled_df.loc[row + 1, "Contains Aus. Names"] = next_row_contains_aus_names

wrangled_df = wrangled_df[wrangled_df["Contains Aus. Names"] == False]
wrangled_df = wrangled_df.drop(columns=["Contains Aus. Names"])

glider_values = [list(row) for row in list(wrangled_df.values)]
glider_values = list(set([val for row in glider_values for val in row]))

glider_final_df = pd.DataFrame(glider_values)
glider_final_df = glider_final_df[glider_final_df[0].notna()]
glider_final_df = glider_final_df.rename(columns={0: 'name'})
glider_final_df

,0
0,Plane Glider
1,MKTV Parafoil
3,Cloud Glider
4,Parachute
5,Wario Wing
6,Peach Parasol
7,Hylian Kite
8,Super Glider
9,Gold Glider
10,Paper Glider


Now we do the wheels

In [30]:
rows = []
rows_simplified = []

for table in wheel_tables:
    data = table.find("tbody")
    for row in data.find_all("tr"):
        if row:
            child = row.find("td")
            grandchild = child.find()
            if grandchild:
                if grandchild.name != "span" and grandchild.name != "div":
                    rows.append(row)
            else:
                rows.append(row)

for row in rows:
    row_simplified = []
    for child in row.children:
        if isinstance(child, NavigableString):
            continue
        
        # If there is an alternative name, it will be in small italics
        # Check for this
        small = child.select('small')
        if small:
            value = child.text.rstrip("\n*/ ").lstrip("\n ") + " [AUS]"
        else:
            value = child.text.rstrip("\n*/ ").lstrip("\n ")

        if value in {"", "New"}:
            row_simplified.append(np.nan)
        else:
            row_simplified.append(value)
    
    rows_simplified.append(row_simplified)

wheel_df = pd.DataFrame(rows_simplified)

wheel_df = wheel_df.dropna(axis=1, how="all")
wheel_df = wheel_df.dropna(axis=0, how="all")

wheel_df = wheel_df.reset_index(drop=True)

wrangled_df = wheel_df.copy()

for row in range(0, len(wheel_df) - 1):
    next_row_contains_aus_names = False
    for col in wrangled_df.columns:
        value = wrangled_df.loc[row, col]
        next_value = wrangled_df.loc[row + 1, col]

        if isinstance(value, str) and isinstance(next_value, str):
            if "[AUS]" in next_value:
                next_row_contains_aus_names = True
                wrangled_df.loc[row, col] = next_value.replace("[AUS]", "")

    # Fill the first row with "False", since it won't have Aus. names
    if row == 0:
        wrangled_df.loc[row, "Contains Aus. Names"] = False

    wrangled_df.loc[row + 1, "Contains Aus. Names"] = next_row_contains_aus_names

wrangled_df = wrangled_df[wrangled_df["Contains Aus. Names"] == False]
wrangled_df = wrangled_df.drop(columns=["Contains Aus. Names"])

wheel_values = [list(row) for row in list(wrangled_df.values)]
wheel_values = list(set([val for row in wheel_values for val in row]))

wheel_final_df = pd.DataFrame(wheel_values)
wheel_final_df = wheel_final_df[wheel_final_df[0].notna()]
wheel_final_df = wheel_final_df.rename(columns={0: 'name'})
wheel_final_df

,0
0,Funky Monster
2,Crimson Slim
3,Ancient Tyres
4,Triforce Tyres
5,Slick
6,Metal
7,GLA Wheels
8,Cushion
9,Gold Wheels
10,Sponge


We don't need gliders or wheels for MK World, so we'll just save now

In [ ]:
glider_final_df.to_csv(wd / 'data' / 'gliders.csv', index=False)
wheel_final_df.to_csv(wd / 'data' / 'wheels.csv', index=False)

## Mario Kart World

Now let's get the Mario Kart World vehicles

In [110]:
url = "https://www.ign.com/wikis/mario-kart-world/All_Vehicles_List_and_Stats_Explained"

driver = webdriver.Firefox()
driver.get(url)

mkw_html = driver.page_source

mkw_soup = BeautifulSoup(mkw_html)

driver.quit()

In [116]:
elements = mkw_soup.find_all(class_="jsx-3850324473 jsx-3179845851 jsx-28683165 wiki-section wiki-html")

all_tables = []

for elem in elements:
    tables = elem.find_all("table")
    all_tables.extend(tables)

print(f"{len(all_tables)} tables found")


1 tables found


Found only one table, which is the one we want, so let's use that one (conveniently also uses the European/Australian names)

In [162]:
table = all_tables[0]

table = table.find("tbody")

rows = table.find_all("tr")

all_values = []

for row in rows:
    values = [val.text.rstrip("\n\t") for val in row.children]
    all_values.append(values)

mkw_vehicle_df = pd.DataFrame(all_values)

mkw_vehicle_df = mkw_vehicle_df[(mkw_vehicle_df[0] != "") & 
                                (mkw_vehicle_df[0] != "Available at the start") &
                                (mkw_vehicle_df[0] != "Unlocked with coins")]

mkw_vehicle_values = [list(row) for row in list(mkw_vehicle_df.values)]
mkw_vehicle_values = list(set([val for row in mkw_vehicle_values for val in row]))

mkw_final_df = pd.DataFrame(mkw_vehicle_values)
mkw_final_df = mkw_final_df[mkw_final_df[0].notna()]
mkw_final_df.columns = ['vehicle_name']

print(f"{len(mkw_final_df)} vehicles found")
mkw_final_df.sort_values(by='vehicle_name')

41 vehicles found


,vehicle_name
3,B Dasher
32,Baby Blooper
21,Big Horn
5,Billdozer
34,Blastronaut III
17,Bowser Bruiser
24,Buggybud
11,Bumble V
26,Carpet Flyer
1,Chargin' Truck


There are 41 results, but there should only be 40. Let's compare all the strings so we can see which are the most similar in case there are a few duplicates

In [139]:
import Levenshtein

In [154]:
tmp_df1 = mkw_final_df.copy()
tmp_df2 = mkw_final_df.copy()

tmp_df1['key'] = 0
tmp_df2['key'] = 0

full_df = tmp_df1.merge(tmp_df2, on='key', how='outer')

full_df = full_df.drop(columns=['key'])

full_df['sim'] = full_df.apply(lambda row: Levenshtein.ratio(row['vehicle_name_x'], row['vehicle_name_y']), axis=1)

full_df[full_df['sim'] < 1].sort_values(by='sim', ascending=False)

,vehicle_name_x,vehicle_name_y,sim
446,R.O.B. H.O.G.,R.O.B.H.O.G,0.916667
1486,R.O.B.H.O.G,R.O.B. H.O.G.,0.916667
1350,Rally Kart,Rallygator,0.700000
1590,Rallygator,Rally Kart,0.700000
408,Standard Kart,Standard Bike,0.692308
...,...,...,...
965,Buggybud,Pipe Frame,0.000000
962,Buggybud,Dolphin Dasher,0.000000
972,Buggybud,Fin Twin,0.000000
967,Buggybud,Reel Racer,0.000000


Looks like there's a duplicate in "R.O.B.H.O.G" and "R.O.B. H.O.G"

In [163]:
mkw_deduplicate_df = mkw_final_df.copy()

mkw_deduplicate_df = mkw_deduplicate_df[mkw_deduplicate_df["vehicle_name"] != "R.O.B. H.O.G."]

mkw_deduplicate_df = mkw_deduplicate_df.reset_index(drop=True)

print(f"{len(mkw_deduplicate_df)} vehicles found")
mkw_deduplicate_df

40 vehicles found


,vehicle_name
0,Tune Thumper
1,Chargin' Truck
2,Loco Moto
3,B Dasher
4,Zoom Buggy
5,Billdozer
6,Cute Scoot
7,Dread Sled
8,Lobster Roller
9,Standard Kart


Now we have the right number.

Now let's combine the two lots into one dataframe that we can save

In [167]:
mk8_df = body_final_df.copy()
mkw_df = mkw_deduplicate_df.copy()

mk8_df.columns = ['vehicle_name']

mk8_df['game'] = 'Mario Kart 8 Deluxe'
mkw_df['game'] = 'Mario Kart World'

mk_vehicles = pd.concat([mk8_df, mkw_df])

mk_vehicles.to_csv(wd / 'data' / 'vehicles.csv', index=False)

mk_vehicles

,vehicle_name,game
0,B-Dasher,Mario Kart 8 Deluxe
1,Master Cycle,Mario Kart 8 Deluxe
2,300 SL Roadster,Mario Kart 8 Deluxe
3,Master Cycle Zero,Mario Kart 8 Deluxe
4,Landship,Mario Kart 8 Deluxe
...,...,...
35,R.O.B.H.O.G,Mario Kart World
36,Rally Romper,Mario Kart World
37,Rallygator,Mario Kart World
38,Standard Bike,Mario Kart World
